# Q2 Training a Flow Matching Model on Images

In Q1 you trained a flow matching model that turns 2D noise into points on a Swiss roll. In this notebook you will train a class-conditional flow matching model on real images: [AFHQ](https://arxiv.org/abs/1912.01865), 64 x 64 photos of cats, dogs and wild animals. The training algorithm is the same as in Q1. In fact, you will **reuse your Q1 code** (`interpolate`, `get_loss` and `sample` in `flow.py`) without changes. What changes is the network, a UNet that also knows the class of the image, and a few training details. You will:

1. implement the missing pieces of the network, the loss with label dropout, weight averaging and classifier-free guidance, and test them on the CPU,
2. train several models on AFHQ on a GPU, in a controlled experiment with one hyperparameter,
3. choose the best run with validation images that the models never saw during training, report it, and look at its samples.

## Workflow

The notebook has three parts:
1. **Part A (CPU)**: implement the TODOs in `flow_unet.py`. The tests take a few seconds on a laptop. Since Q2 builds on your Q1 code, your Q1 flow matching tests must pass too. **Make sure all tests pass before you train.**
2. **Training (GPU)**: train at least 3 models on AFHQ with the provided `train_flow.py`, which uses your code. On Delta AI, submit one batch job per run with `train_flow.sbatch` (see the README); elsewhere, run them from this notebook. On Delta AI, a 10,000-step run takes about 8 minutes (on an RTX 3060, about 13 minutes). Every run starts a new model, and saves checkpoints, so if your job or session ends early, you can continue the run with `--resume`.
3. **Part B (GPU or Mac)**: compare the losses of your runs, and look at the samples of your best model. This takes about a minute on a GPU. It loads your models from their checkpoints, so you can do it any time after training. On Delta AI, it runs as a batch job (see Part B).

The code picks the best available device automatically. Your options:
- **Delta AI (recommended)**: as in MP3 and MP4, see the README and `DeltaAI.md`: train with a batch job, and run Part B with a second batch job that runs this whole notebook,
- **your own NVIDIA GPU** (`cuda`),
- **an Apple Silicon Mac** (`mps`): fine for Part B, but training is much slower than on an NVIDIA GPU, so we recommend training on Delta AI.

Part B is slow on a CPU (minutes per figure). Every figure in Part B is also saved to `outputs/`.

As in Q1, either enable the autoreload extension or restart the kernel after updating `.py` files.

## Convention and Notation

We use the same convention and notation as in Q1: $t = 0$ is noise $\epsilon \sim \mathcal{N}(0, I_d)$, $t = 1$ is data $z$, $x_t = t\,z + (1 - t)\,\epsilon$, and the network predicts the velocity $z - \epsilon$. Now $z$ is an image with a class label $y \in \{0 = \text{cat}, 1 = \text{dog}, 2 = \text{wild}\}$, and $u_t^\theta(x \mid y)$ is the velocity predicted for class $y$. The *null* label $\varnothing$ stands for "any class".

In [ ]:
# Uncomment the following pip command to install requirements
# NOT needed on Delta AI: the batch jobs use the course virtual env
# ! pip install -r requirements.txt

In [ ]:
# loading autoreload extension for jupyter
%load_ext autoreload
try:
    # IPython 9+: always reload whole modules (its new partial reload can keep running your old code)
    %autoreload 2 --full
except Exception:
    # older IPython (e.g. on Delta AI) has no --full and always reloads whole modules
    %autoreload 2

In [ ]:
import glob
import os
import unittest

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torchvision.utils import save_image

from q2_helpers import show_images


import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Part A: Building Blocks (CPU)

## Reusing Q1

`ImageFlowMatching` in `flow_unet.py` is a subclass of your `FlowMatching` from Q1. It inherits `interpolate`, `get_loss` and `sample` without changes, and only overrides what is different for images:
- `get_velocity(x, t, labels, scale)`: instead of an MLP on $[x, t]$, a UNet (`FlowUNet`) predicts the velocity from the image, the time and the class label, optionally with classifier-free guidance (Q2.6),
- `sample_times`: logit-normal instead of uniform training times (provided, see Training below),
- `get_loss(z, labels)`: drops some of the labels, then calls your Q1 `get_loss` (Q2.4).

Your Q1 methods work for images because they handle data of any shape and pass their extra arguments on to `get_velocity`. The cell below re-runs the Q1 flow matching tests. Make sure they pass before you continue.

In [ ]:
suite = unittest.TestLoader().loadTestsFromNames(['test_functions.TestClass.test_flow_interpolate',
                                                  'test_functions.TestClass.test_flow_loss',
                                                  'test_functions.TestClass.test_flow_sample'])
unittest.TextTestRunner().run(suite)

## The Network

In Q1, the velocity network was an MLP that took $t$ as an extra input coordinate. For images we use a **UNet** (`FlowUNet` in `flow_unet.py`). It processes the image at four resolutions (64, 32, 16 and 8 pixels), with residual blocks at every resolution, self-attention at the lowest resolution, and skip connections from the downsampling path to the upsampling path, so that the output keeps the spatial details of the input. The layers are provided. You will implement how the network knows **which time** $t$ and **which class** $y$ it should predict the velocity for (Q2.1 and Q2.2), and how its blocks are connected (Q2.3).

The time and the class are first turned into one embedding vector per image, `emb = time_emb(t) + class_emb(y)`:
- **Time embedding.** A scalar $t$ is a poor input for a network: the velocity changes a lot between $t = 0.98$ and $t = 0.99$, but a linear layer sees almost the same input. As for the positions in a Transformer, we instead represent $t$ by sines and cosines at many frequencies, from slow to fast, and pass them through a small MLP.
- **Class embedding.** `class_emb` is a lookup table (`nn.Embedding`) with one learned vector per class, plus one extra vector for the **null label** $\varnothing$ (`num_classes`). The null label is what makes classifier-free guidance possible with a single network (see the loss below).

**Q2.1 [1 pts Autograded]** Implement `TimeEmbedding.sinusoidal` in `flow_unet.py`, following its docstring. Once you have implemented the function, run the following cell to test your implementation.

In [ ]:
suite = unittest.TestLoader().loadTestsFromName('test_functions.TestClass.test_time_embedding')
unittest.TextTestRunner().run(suite)

Every residual block then uses the embedding to **modulate** its features, with *feature-wise linear modulation* (FiLM, [Perez et al., 2018](https://arxiv.org/abs/1709.07871)): a linear layer predicts one scale $\gamma_k$ and one shift $\beta_k$ per channel $k$, and the normalized features of channel $k$ are transformed as

$$
h_k \leftarrow \text{norm}(h)_k \cdot (1 + \gamma_k) + \beta_k ,
$$

with the same $\gamma_k, \beta_k$ for every pixel. So the time and the class can turn channels up, down, or off in every block of the network. We use $1 + \gamma_k$ rather than $\gamma_k$, so that $\gamma = \beta = 0$ leaves the features unchanged.

**Q2.2 [2 pts Autograded]** Implement `ResBlock.forward` and `FlowUNet.embed` in `flow_unet.py`, following their docstrings. Once you have implemented the functions, run the following cell to test your implementation.

In [ ]:
suite = unittest.TestLoader().loadTestsFromName('test_functions.TestClass.test_resblock_film')
unittest.TextTestRunner().run(suite)

## The UNet

`FlowUNet.forward` connects the blocks into a UNet:
- the **encoder** processes the image at four resolutions (64, 32, 16 and 8 pixels), halving the resolution with a strided convolution before every level but the first,
- the **middle** works at the lowest resolution, with self-attention, so that every position can see the whole image,
- the **decoder** mirrors the encoder: it doubles the resolution before every level but the first, and every level first concatenates the encoder output of the same resolution (a *skip connection*), which brings back the fine details lost by downsampling,
- a final normalization, SiLU and convolution map the features to the 3 channels of the velocity.

All layers are created in `FlowUNet.__init__` (provided): read it to see which module is used where.

**Q2.3 [2 pts Autograded]** Implement `FlowUNet.forward` in `flow_unet.py`, following its docstring. The test uses reference versions of your Q2.1 and Q2.2 code, so it only checks how you connect the blocks. Once you have implemented the function, run the following cell to test your implementation.

In [ ]:
suite = unittest.TestLoader().loadTestsFromName('test_functions.TestClass.test_unet_forward')
unittest.TextTestRunner().run(suite)

## Training

We train with your conditional flow matching loss from Q1.2, now with images $z$ and their class labels $y$, so the network predicts $u_t^\theta(x_t \mid y)$.

There are two differences from Q1.2:
- **Training times.** Like Stable Diffusion 3, we draw $t = \text{sigmoid}(n)$ with $n \sim \mathcal{N}(0, 1)$ (a *logit-normal* distribution) instead of $t \sim \text{Unif}[0, 1]$. This trains more on intermediate times and less on $t$ close to $0$ or $1$, which [Esser et al., 2024](https://arxiv.org/abs/2403.03206) found to give better models. The provided `ImageFlowMatching.sample_times` does this, and your Q1 `interpolate` picks it up automatically, since it draws its times with `self.sample_times`.
- **Label dropout.** Classifier-free guidance (below) needs the *unconditional* velocity $u_t^\theta(x \mid \varnothing)$ as well. Instead of training a second network, we replace the label of each image by the null label with probability $p_\text{uncond}$ (10%). The same network then learns both the velocity for a given class and the velocity for "any class".

**Q2.4 [2 pts Autograded]** Implement `ImageFlowMatching.get_loss` in `flow_unet.py`: drop the labels, then reuse your Q1 loss. The test also checks that your Q1 `interpolate` and `get_loss` work for images. Once you have implemented the function, run the following cell to test your implementation.

In [ ]:
suite = unittest.TestLoader().loadTestsFromName('test_functions.TestClass.test_image_flow_loss')
unittest.TextTestRunner().run(suite)

**Exponential moving average (EMA).** With stochastic gradients, the weights keep jumping around a good solution instead of settling on it, so the weights of a single step are worse than an *average* of the weights over the last steps. Flow models therefore keep a second copy of the network whose weights are an exponential moving average of the trained weights, updated after every optimizer step:

$$
\theta_\text{EMA} \leftarrow \beta\, \theta_\text{EMA} + (1 - \beta)\, \theta ,
$$

with a decay $\beta$ close to 1. The EMA averages over roughly the last $1 / (1 - \beta)$ steps (`--ema_decay` in the training script below). The EMA weights are never trained directly, and they are the ones we evaluate and sample with.

**Q2.5 [1 pts Autograded]** Implement `update_ema` in `flow_unet.py`. Once you have implemented the function, run the following cell to test your implementation.

In [ ]:
suite = unittest.TestLoader().loadTestsFromName('test_functions.TestClass.test_update_ema')
unittest.TextTestRunner().run(suite)

## Classifier-Free Guidance

In practice, samples look much more like their class with **classifier-free guidance (CFG)** ([Ho & Salimans, 2022](https://arxiv.org/abs/2207.12598)). CFG combines the guided velocity $u_t^\theta(x \mid y)$ with the unconditional velocity $u_t^\theta(x \mid \varnothing)$, and moves further in the direction in which the class changes the prediction:

$$
\tilde{u}_t^\theta(x \mid y) = u_t^\theta(x \mid \varnothing) + w \left( u_t^\theta(x \mid y) - u_t^\theta(x \mid \varnothing) \right),
$$

where $w$ is the **guidance scale**. With $w = 0$ we get the unconditional velocity, with $w = 1$ the guided velocity, and with $w > 1$ we extrapolate. Note that guidance doubles the cost of each sampling step, since it needs two predictions. Text-to-image models such as Stable Diffusion 3 use exactly the same formula, with a text prompt as the condition and the empty prompt as $\varnothing$.

Because your Q1 `sample` passes its extra arguments on to `get_velocity`, `flow.sample(noise, labels, w)` samples with guidance.

**Q2.6 [1 pts Autograded]** Implement `ImageFlowMatching.get_velocity` in `flow_unet.py`. With $w = 1$ (as in training), call the network only once. Once you have implemented the function, run the following cell to test your implementation.

In [ ]:
suite = unittest.TestLoader().loadTestsFromName('test_functions.TestClass.test_image_cfg_velocity')
unittest.TextTestRunner().run(suite)

Before you train, run all tests at once and make sure they pass.

In [ ]:
suite = unittest.TestLoader().loadTestsFromNames(['test_functions.TestClass.test_flow_interpolate',
                                                  'test_functions.TestClass.test_flow_loss',
                                                  'test_functions.TestClass.test_flow_sample',
                                                  'test_functions.TestClass.test_time_embedding',
                                                  'test_functions.TestClass.test_resblock_film',
                                                  'test_functions.TestClass.test_unet_forward',
                                                  'test_functions.TestClass.test_image_flow_loss',
                                                  'test_functions.TestClass.test_update_ema',
                                                  'test_functions.TestClass.test_image_cfg_velocity'])
unittest.TextTestRunner(verbosity=2).run(suite)

# Training Your Models (GPU)

We now train `FlowUNet`s (about 11M parameters) on AFHQ with the provided `train_flow.py`, which uses your code from Q1 and Q2.1-Q2.6.

**Data.** On Delta AI, the data is already prepared for you, so skip the next cell. Elsewhere, the next cell downloads [AFHQv2](https://huggingface.co/datasets/huggan/AFHQv2) and saves it at 64 x 64 to `data/afhq64.pt` (~200 MB). This takes about 10 minutes and ~1 GB of free disk space, and you only need to do it once. 64 images per class are held out for validation: they are never used for training, and you use them to compare your runs and choose the best hyperparameters.

**The training script.** Every step draws a random batch of training images (flipped horizontally at random), computes your loss with label dropout, takes an Adam step, and updates the EMA weights. Every 100 steps, it records the training loss, and every `--sample_every` steps, it prints the training loss and the validation loss of the EMA weights (see Q2.7), and saves sample grids and a checkpoint to `--run_dir`. Every run starts a new model. If a run stops early, run the same command with `--resume` added: it continues the run in `--run_dir` from its last checkpoint, with the hyperparameters it was started with.

**A controlled experiment (Q2.7).** The default hyperparameters train, but they are **not tuned**. Instead of guessing better ones, you will run a **controlled experiment**: pick **one** hyperparameter from the table below, and train **at least 3 runs** that differ **only** in its value. Everything else, including `--max_steps`, stays the same, so that any difference between the runs is caused by that hyperparameter. Give every run its own `--run_dir`, named after its value.

| flag | default | meaning |
|---|---|---|
| `--batch_size` | 32 | images per step (lower it if you run out of memory) |
| `--lr` | 1e-4 | peak learning rate of Adam |
| `--lr_schedule` | `constant` | `constant`, or `cosine` decay to 0 at `--max_steps` |
| `--warmup_steps` | 0 | linear warmup of the learning rate from 0 |
| `--grad_clip` | 0 | maximum gradient norm (0: no clipping) |
| `--ema_decay` | 0.99 | the EMA averages over roughly the last 1 / (1 - `ema_decay`) steps |
| `--max_steps` | 10000 | length of the run |
| `--channels` | `32,64,128,256` | width of the network |
| `--p_uncond` | 0.1 | probability of dropping the label |

**Choosing your experiment.** Think about which hyperparameter most likely limits a short run, and choose values that are far enough apart to show a clear effect. Include the default value as one of them, so that you see how much you improve on the defaults. A 10,000-step run takes about 8 minutes on Delta AI (about 13 minutes on an RTX 3060), and every run stops after at most 40 minutes (`--max_minutes`). (If `--run_dir` already has a run, the script never overwrites it: it starts the new run in `<run_dir>_2`, `<run_dir>_3`, ..., and prints which folder it uses.)

**Budget.** Use at most 10,000 steps per run, and at most 6 runs in total (about 50 GPU-minutes on Delta AI).

**Your goal.** With the default hyperparameters, a 10,000-step run reaches a validation loss of about 0.166. Aim for a best run at about **0.156 or lower** within 10,000 steps: its samples in Q2.8 and Q2.9 are clearly cats, dogs and wild animals, and sharper than with the defaults. This target is not graded, and it is only a guide: two runs with the same hyperparameters can differ by about 0.001. If your first experiment does not reach it, you may run a second experiment on another hyperparameter, with the first one set to its best value. Your best run so far is then one of the runs of the second experiment (it has the default value of the new hyperparameter), so it needs only 2 new runs. Report both experiments in Q2.7. If a run with the default hyperparameters ends far above 0.166 (e.g. above 0.17), something is probably wrong, e.g. a mistake in your Q2 code that the tests did not catch.

**On Delta AI (recommended)**, submit one batch job per run from the MP folder on the login node: the runs then train at the same time. The jobs activate the course virtual env by themselves, and flags after `train_flow.sbatch` are passed on to `train_flow.py`. For example, this experiment on gradient clipping (an example of the syntax, not a recommendation) submits 3 jobs:
```bash
for v in 0 0.5 1; do sbatch --export=ALL,RUN_DIR=runs/grad_clip$v train_flow.sbatch --grad_clip $v; done
```
Each job trains for at most 40 minutes; to continue a run, submit it again with `--resume` (e.g. `sbatch --export=ALL,RUN_DIR=runs/grad_clip0.5 train_flow.sbatch --resume`). `squeue -u $USER` lists your jobs, and `tail -f ece494mp5-flow_afhq-<job id>.out` shows the progress.

**Elsewhere**, run the cell below (change it for your experiment). It trains the runs one after the other. Add `--resume` to continue a run. On Delta AI, it does nothing.

In [ ]:
from q2_helpers import DATA_PATH, prepare_afhq

if os.path.exists(DATA_PATH):
    print(f'{DATA_PATH} already exists')
else:
    prepare_afhq(DATA_PATH)  # only once: downloads ~7 GB in parts, saves ~200 MB, takes about 10 minutes

In [ ]:
from q2_helpers import DELTA_DATA_PATH

if os.path.exists(DELTA_DATA_PATH):
    print('On Delta AI, train with train_flow.sbatch instead (see above)')
else:
    FLAG, VALUES = 'grad_clip', ['0', '0.5', '1']  # your experiment (this one is only an example): one run per value
    for v in VALUES:
        %run train_flow.py --run_dir runs/{FLAG}{v} --{FLAG} {v}

# Part B: Evaluating Your Trained Models (GPU or Mac)

Once your runs have finished, list them in `EXPERIMENT_RUNS` in the cell below. Q2.7 compares them, and Q2.8 and Q2.9 use the run with the lowest validation loss (or set `RUN_DIR` to another run). Part B only needs the checkpoints of your runs, so you can also start here in a new session.

**On Delta AI**, edit the cell below in this notebook on your computer, upload the notebook to your MP folder (e.g. with `scp`), and run it as a batch job from the login node:
```bash
sbatch run_notebook.sbatch Q2_flow_matching.ipynb
```
It runs the whole notebook in about a minute: the tests of Part A, no training (that is what `train_flow.sbatch` is for), and Part B. Then download the figures in `outputs/` for your report. `Q2_flow_matching_run.ipynb` is a copy of this notebook with all its outputs, and the job log (`ece494mp5-mp5_notebook-<job id>.out`) shows any errors.

In [ ]:
from flow_unet import ImageFlowMatching
from q2_helpers import get_device, load_trained_model, randn

EXPERIMENT_RUNS = []  # the run folders of your experiment (Q2.7), e.g. ['runs/grad_clip0', 'runs/grad_clip0.5', 'runs/grad_clip1']
RUN_DIR = None  # the run for Q2.8 and Q2.9; None: the run of EXPERIMENT_RUNS with the lowest validation loss
DEVICE = get_device()  # 'cuda' (NVIDIA GPU), 'mps' (Apple Silicon GPU), or 'cpu' if no GPU is found
print(f'Using device: {DEVICE}')
N_STEPS = 50   # Euler steps from t = 0 (noise) to t = 1 (data)
SCALE = 2.0    # default guidance scale (see Q2.9)
os.makedirs('outputs', exist_ok=True)

## Q2.7 Controlled Experiment

To check how well a model learned the distribution of the images, we evaluate it on validation images that were never used for training. `val_loss` in `q2_helpers.py` computes the conditional flow matching loss of the EMA weights on the 192 validation images (64 per class), with the true labels and without guidance ($w = 1$). It uses **fixed** noise and **fixed** times $t \in \{0.1, 0.3, 0.5, 0.7, 0.9\}$ instead of random ones, so the same weights always get the same value, and runs can be compared fairly. Lower is better. Since the model never saw these images, a low validation loss means that it generalizes to new images of cats, dogs and wild animals, and did not just memorize its training images.

You use the validation loss to choose the best value of your hyperparameter. The training loss cannot tell you this, and it is not on the same scale: it uses random times (from `sample_times`), label dropout and the current (not EMA) weights. So compare the runs within each plot, not the two plots with each other.

The cell below reads the logs of the runs in `EXPERIMENT_RUNS` from their checkpoints. It prints which hyperparameters differ between the runs (exactly one should), plots their training and validation losses, and prints a table with the validation loss of each run and the hyperparameters of the best run. If you ran a second experiment, run the cell once per experiment, with `EXPERIMENT_RUNS` set to its runs (the figure is saved under the name of the hyperparameter). Run the experiment with your best run last, since Q2.8 and Q2.9 use the best run of `EXPERIMENT_RUNS`.

In [ ]:
assert EXPERIMENT_RUNS, 'List the run folders of your experiment in EXPERIMENT_RUNS (first cell of Part B)'
logs = {run: load_trained_model(run)[1] for run in EXPERIMENT_RUNS}
NOT_HYPERPARAMETERS = {'data_path', 'run_dir', 'max_minutes', 'sample_every', 'resume'}
varied = [k for k in logs[EXPERIMENT_RUNS[0]]['args'] if k not in NOT_HYPERPARAMETERS
          and len({str(log['args'].get(k)) for log in logs.values()}) > 1]
print(f'Hyperparameters that differ between the runs: {", ".join(varied) or "none"}')
if len(varied) != 1:
    print('WARNING: the runs of a controlled experiment differ in exactly one hyperparameter!')


def setting(log):
    return ', '.join(f'{k} = {log["args"][k]}' for k in varied)


fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for run, log in logs.items():
    for ax, key in zip(axes, ['losses', 'val_losses']):
        steps, values = zip(*log[key])
        ax.plot(steps, values, label=f'{run} ({setting(log)})')
axes[0].set_yscale('log')  # the training loss drops fast at first
axes[0].yaxis.set_major_formatter('{x:.2f}')
axes[0].yaxis.set_minor_formatter('{x:.2f}')
for ax, title in zip(axes, ['Training loss (average over 100 steps)', 'Validation loss (EMA weights)']):
    ax.set_title(title)
    ax.set_xlabel('training step')
    ax.grid(True, which='both', alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.savefig(f'outputs/q2_7_experiment_{"_".join(varied) or "runs"}.png', dpi=100)
plt.show()

print(f'{"run":<24} {"setting":<20} {"steps":>6} {"minutes":>7} {"final val loss":>15} {"lowest val loss (step)":>23}')
for run, log in logs.items():
    best_step, best = min(log['val_losses'], key=lambda h: h[1])
    print(f'{run:<24} {setting(log):<20} {log["step"]:>6} {log["minutes"]:>7.1f} {log["val_losses"][-1][1]:>15.4f} '
          f'{best:>15.4f} ({best_step})')
best_run = min(logs, key=lambda run: logs[run]['val_losses'][-1][1])
print(f'Lowest final validation loss: {best_run} ({logs[best_run]["val_losses"][-1][1]:.4f}), trained with')
for k, v in logs[best_run]['args'].items():
    if k not in NOT_HYPERPARAMETERS:
        print(f'  --{k} {v}')

**Q2.7 [4 pts Manual graded]** Upload the figure and the table above (one of each per experiment, if you ran two), and answer the following:
1. Which hyperparameter did you vary, with which values? What did you keep fixed?
2. How does the hyperparameter change the training and validation loss curves (how fast they drop, how low they get, how noisy they are)? Explain why.
3. Which run do you choose, based on the validation loss? Report its hyperparameters (all of them, as printed above) and its validation loss. Did it reach about 0.156 (see the Training section)? Based on your results, what would you change next to lower it further, and why?

## Your Best Model

The cell below loads `model`, the EMA weights of `RUN_DIR` (by default, your run with the lowest validation loss), which Q2.8 and Q2.9 use. It is an `ImageFlowMatching` model, so `model.sample(noise, label, w)` samples with your Q1 `sample` and guidance scale `w`.

In [ ]:
RUN_DIR = RUN_DIR or best_run
ema_net, log = load_trained_model(RUN_DIR, DEVICE)  # EMA weights
model = ImageFlowMatching(ema_net)
CLASSES = log['classes']
print(f"Loaded {RUN_DIR} ({sum(p.numel() for p in ema_net.parameters()) / 1e6:.1f}M parameters) after {log['step']} "
      f"training steps ({log['minutes']:.0f} minutes), validation loss {log['val_losses'][-1][1]:.4f}")


def generate(flow, label, n, seed=0, scale=SCALE, n_steps=N_STEPS):
    """Generate n images of class `label` from pure noise, with classifier-free guidance."""
    return flow.sample(randn(n, seed, DEVICE), label, scale, n_steps=n_steps)[-1].cpu()

## Q2.8 Training

The cell below shows the samples that the training loop saved at log-spaced steps (EMA weights, one row per class).

In [ ]:
files = sorted(glob.glob(os.path.join(RUN_DIR, 'samples', 'step*_ema.png')))
# log-spaced steps (e.g. 1k, 2k, 5k, 13k, 30k), since the samples change the most early in training
picks = [files[i - 1] for i in np.unique(np.geomspace(1, len(files), 5).round().astype(int))]
fig, axes = plt.subplots(len(picks), 1, figsize=(8, 3.2 * len(picks)), squeeze=False)
for ax, f in zip(axes[:, 0], picks):
    ax.imshow(Image.open(f))
    ax.set_title(f'step {int(os.path.basename(f)[4:10])}')
    ax.axis('off')
plt.tight_layout()
plt.savefig('outputs/q2_8_progress.png', dpi=100)
plt.show()

**Q2.8 [1 pts Manual graded]** Upload the visualization above and answer: how do the samples change during training? What does the model learn first, and what last?

## Q2.9 Guidance

Finally, we generate images of each class from pure noise with several guidance scales $w$. Within each column, all images start from the same noise.

In [ ]:
CFG_SCALES = [0, 1, 4, 16]
for c, name in enumerate(CLASSES):
    images = torch.cat([generate(model, c, 8, seed=10, scale=w) for w in CFG_SCALES])
    show_images(images, ncols=8, row_labels=[f'{name}, w = {w}' for w in CFG_SCALES], fname=f'q2_9_cfg_{name}.png')

**Q2.9 [2 pts Manual graded]** Upload the visualizations above and answer the following:
1. Compare the samples for $w = 0$, $1$ and $4$. What happens at $w = 0$, and why? How do the quality and the diversity of the samples change as $w$ increases? Explain using the CFG equation.
2. What goes wrong when $w$ is too large (look at $w = 16$)? Which guidance scale would you choose for this model, and why?